In [1]:
# ============================================================
# PMSM ELECTRO-THERMAL PHYSICS DEFENSIBILITY ANALYSIS
# ============================================================
#
# Dataset:
# Electric Motor Temperature / PMSM measurements
#
# Expected columns:
# u_q, coolant, stator_winding, u_d, stator_tooth,
# motor_speed, i_d, i_q, pm, stator_yoke, ambient,
# torque, profile_id
#
# IMPORTANT:
# This script does NOT assume a timestamp column exists.
# Derivatives are calculated within profile_id using row order.
# If the dataset rows are not temporally ordered within each
# profile, derivative-based conclusions should NOT be treated
# as strong physical evidence.
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIGURATION
# ============================================================

DATA_PATH = "measures_v2.csv"

# If running locally, change DATA_PATH, for example:
# DATA_PATH = "measures_v2.csv"

OUTPUT_DIR = "physics_analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROFILE_COL = "profile_id"

# Actual dataset columns
UD = "u_d"
UQ = "u_q"
ID = "i_d"
IQ = "i_q"

SPEED = "motor_speed"
TORQUE = "torque"

TW = "stator_winding"
TT = "stator_tooth"
TY = "stator_yoke"
TPM = "pm"

COOLANT = "coolant"
AMBIENT = "ambient"

# ============================================================
# 1. LOAD DATA
# ============================================================

print("=" * 80)
print("PMSM ELECTRO-THERMAL PHYSICS DEFENSIBILITY ANALYSIS")
print("=" * 80)

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# 2. BASIC VALIDATION
# ============================================================

required_columns = [
    PROFILE_COL,
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    TORQUE,
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"\nMissing required columns:\n{missing}\n"
        f"\nAvailable columns:\n{df.columns.tolist()}"
    )

print("\nAll required columns are present.")

# ============================================================
# 3. NUMERIC CLEANING
# ============================================================

numeric_columns = [
    UD, UQ, ID, IQ,
    SPEED, TORQUE,
    TW, TT, TY, TPM,
    COOLANT, AMBIENT
]

before_rows = len(df)

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[PROFILE_COL] = pd.to_numeric(
    df[PROFILE_COL],
    errors="coerce"
)

df = df.dropna(
    subset=required_columns
).copy()

after_rows = len(df)

print(
    f"\nRows removed because of NaN/non-numeric values: "
    f"{before_rows - after_rows}"
)

print(f"Remaining rows: {after_rows:,}")

# ============================================================
# 4. PROFILE INFORMATION
# ============================================================

profiles = sorted(df[PROFILE_COL].unique())

print(f"\nNumber of profiles: {len(profiles)}")

profile_counts = (
    df.groupby(PROFILE_COL)
      .size()
      .sort_values()
)

print("\nProfile size statistics:")
print(profile_counts.describe())

# ============================================================
# 5. CHECK PROFILE ORDERING
# ============================================================
#
# There is NO explicit timestamp column in the supplied data.
#
# Therefore:
#   derivative = temperature.diff()
#
# assumes the CSV row order within profile_id corresponds
# to temporal progression.
#
# We explicitly document this assumption.
# ============================================================

print("\n" + "=" * 80)
print("TEMPORAL ORDERING CHECK")
print("=" * 80)

timestamp_candidates = [
    "timestamp",
    "time",
    "time_stamp",
    "datetime",
    "Date",
    "Time"
]

timestamp_columns = [
    c for c in timestamp_candidates
    if c in df.columns
]

if timestamp_columns:

    print(
        "\nTimestamp-like column found:",
        timestamp_columns
    )

    TIME_COL = timestamp_columns[0]

    print(
        f"Using '{TIME_COL}' as temporal ordering column."
    )

    df[TIME_COL] = pd.to_numeric(
        df[TIME_COL],
        errors="coerce"
    )

    df = (
        df.sort_values(
            [PROFILE_COL, TIME_COL]
        )
        .reset_index(drop=True)
    )

    derivative_basis = "explicit_timestamp"

else:

    TIME_COL = None

    print(
        "\nWARNING:"
    )

    print(
        "No timestamp/time column exists in the supplied dataset."
    )

    print(
        "Derivatives will use CSV row order within each profile."
    )

    print(
        "Therefore dT/dt should be interpreted as "
        "'temperature change per sample', NOT physical dT/dt."
    )

    derivative_basis = "row_order"

# ============================================================
# 6. ELECTRICAL POWER FEATURES
# ============================================================

print("\n" + "=" * 80)
print("ELECTRICAL / MECHANICAL POWER FEATURES")
print("=" * 80)

# dq-axis instantaneous electrical power.
#
# Depending on the dq convention, a scaling factor such as
# 3/2 may be required. We therefore retain the raw dq power
# and avoid claiming absolute physical power unless the
# convention is known.
#
# Raw dq:
# P = ud*id + uq*iq
#
# Common PMSM convention:
# P = 1.5 * (ud*id + uq*iq)
#
# We calculate both.

df["P_dq_raw"] = (
    df[UD] * df[ID]
    +
    df[UQ] * df[IQ]
)

df["P_electrical"] = (
    1.5 * df["P_dq_raw"]
)

# ============================================================
# COPPER LOSS
# ============================================================
#
# Important:
# Without explicit phase resistance and dq convention,
# this is a proxy rather than an exact physical copper loss.
#
# We use the same convention commonly used in the earlier
# pipeline:
#
# P_copper = i_d^2 + i_q^2
#
# This is proportional to I^2 but does NOT have physical watts
# unless multiplied by an appropriate resistance factor.
#
# If Rs is known:
#
# P_copper = Rs * (i_d^2 + i_q^2)
#
# ============================================================

# Set this if you know the actual stator resistance.
# Otherwise leave as None.

RS = None

if RS is not None:

    df["P_copper"] = (
        RS * (
            df[ID] ** 2
            +
            df[IQ] ** 2
        )
    )

    copper_definition = "Rs * (id^2 + iq^2)"

else:

    df["P_copper"] = (
        df[ID] ** 2
        +
        df[IQ] ** 2
    )

    copper_definition = (
        "id^2 + iq^2 current-squared proxy"
    )

# ============================================================
# MECHANICAL POWER
# ============================================================

# rpm -> rad/s
df["omega_rad_s"] = (
    2.0
    * np.pi
    * df[SPEED]
    / 60.0
)

df["P_mechanical"] = (
    df[TORQUE]
    *
    df["omega_rad_s"]
)

print("\nPower features created:")
print("P_dq_raw")
print("P_electrical")
print("P_copper")
print("omega_rad_s")
print("P_mechanical")

# ============================================================
# 7. THERMAL DIFFERENCES
# ============================================================

print("\n" + "=" * 80)
print("THERMAL TEMPERATURE DIFFERENCES")
print("=" * 80)

df["dTw"] = (
    df[TW] - df[COOLANT]
)

df["dTt_y"] = (
    df[TT] - df[TY]
)

df["dTt_pm"] = (
    df[TT] - df[TPM]
)

df["dTy_c"] = (
    df[TY] - df[COOLANT]
)

df["dTy_a"] = (
    df[TY] - df[AMBIENT]
)

thermal_difference_columns = [
    "dTw",
    "dTt_y",
    "dTt_pm",
    "dTy_c",
    "dTy_a"
]

print(
    df[thermal_difference_columns]
    .describe()
    .T[
        ["mean", "std", "min", "max"]
    ]
)

# ============================================================
# 8. THERMAL DERIVATIVES
# ============================================================

print("\n" + "=" * 80)
print("THERMAL DYNAMICS")
print("=" * 80)

# Ensure profile order
df = (
    df.sort_values(
        [PROFILE_COL]
        +
        ([TIME_COL] if TIME_COL else [])
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Derivative per sample / timestep
# ------------------------------------------------------------

df["dTw_dt"] = (
    df.groupby(PROFILE_COL)[TW]
      .diff()
)

df["dTT_dt"] = (
    df.groupby(PROFILE_COL)[TT]
      .diff()
)

df["dTY_dt"] = (
    df.groupby(PROFILE_COL)[TY]
      .diff()
)

df["dTPM_dt"] = (
    df.groupby(PROFILE_COL)[TPM]
      .diff()
)

derivative_columns = [
    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt"
]

# If timestamp exists and is numeric, divide by dt.
if TIME_COL is not None:

    df["_dt"] = (
        df.groupby(PROFILE_COL)[TIME_COL]
          .diff()
    )

    valid_dt = (
        df["_dt"].notna()
        &
        (df["_dt"] > 0)
    )

    for col in derivative_columns:

        df.loc[valid_dt, col] = (
            df.loc[valid_dt, col]
            /
            df.loc[valid_dt, "_dt"]
        )

    df.loc[
        ~valid_dt,
        derivative_columns
    ] = np.nan

    print(
        "\nPhysical derivatives calculated using "
        f"'{TIME_COL}'."
    )

else:

    print(
        "\nDerivatives calculated per sample."
    )

    print(
        "WARNING: No physical time interval is available."
    )

print("\nDerivative columns successfully created.")

# ============================================================
# 9. BASIC PHYSICAL STATISTICS
# ============================================================

print("\n" + "=" * 80)
print("BASIC PHYSICAL STATISTICS")
print("=" * 80)

basic_columns = [
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    TORQUE,
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT,
    "P_copper",
    "P_electrical",
    "P_mechanical"
]

basic_stats = (
    df[basic_columns]
    .describe()
    .T[
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    ]
)

print(basic_stats)

basic_stats.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "basic_physical_statistics.csv"
    )
)

# ============================================================
# 10. THERMAL CORRELATION
# ============================================================

print("\n" + "=" * 80)
print("THERMAL CORRELATION MATRIX")
print("=" * 80)

thermal_columns = [
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT,
    "dTw",
    "dTt_y",
    "dTt_pm",
    "dTy_c",
    "dTy_a",
    "P_copper"
]

thermal_corr = (
    df[thermal_columns]
    .corr()
)

print(
    thermal_corr.round(4)
)

thermal_corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_correlation_matrix.csv"
    )
)

# ============================================================
# 11. ELECTRO-THERMAL CORRELATION
# ============================================================

print("\n" + "=" * 80)
print("ELECTRO-THERMAL CORRELATION MATRIX")
print("=" * 80)

electro_thermal_columns = [
    ID,
    IQ,
    UD,
    UQ,
    SPEED,
    TORQUE,
    "P_copper",
    TW,
    TT,
    TY,
    TPM,
    COOLANT,
    AMBIENT
]

electro_thermal_corr = (
    df[electro_thermal_columns]
    .corr()
)

print(
    electro_thermal_corr.round(4)
)

electro_thermal_corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "electro_thermal_correlation_matrix.csv"
    )
)

# ============================================================
# 12. THERMAL DYNAMIC CORRELATION
# ============================================================

print("\n" + "=" * 80)
print("THERMAL DYNAMIC CORRELATION")
print("=" * 80)

dynamic_columns = [
    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt",
    "P_copper"
]

dynamic_corr = (
    df[dynamic_columns]
    .corr()
)

print(
    dynamic_corr.round(4)
)

dynamic_corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_dynamic_correlation.csv"
    )
)

# ============================================================
# 13. COPPER LOSS -> THERMAL DYNAMICS
# ============================================================

print("\n" + "=" * 80)
print("COPPER LOSS → THERMAL DYNAMICS")
print("=" * 80)

copper_dynamic_results = {}

for col in [
    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt"
]:

    valid = df[
        ["P_copper", col]
    ].dropna()

    corr = (
        valid["P_copper"]
        .corr(valid[col])
    )

    copper_dynamic_results[col] = corr

    print(
        f"P_copper vs {col:10s}: "
        f"{corr: .6f}"
    )

pd.DataFrame(
    copper_dynamic_results,
    index=["correlation"]
).T.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "copper_to_thermal_dynamics.csv"
    )
)

# ============================================================
# 14. THERMAL ORDERING DIAGNOSTIC
# ============================================================

print("\n" + "=" * 80)
print("THERMAL ORDERING DIAGNOSTIC")
print("=" * 80)

thermal_ordering = {
    "Tw > coolant":
        (df[TW] > df[COOLANT]).mean(),

    "Tt > coolant":
        (df[TT] > df[COOLANT]).mean(),

    "Ty > coolant":
        (df[TY] > df[COOLANT]).mean(),

    "Tw > ambient":
        (df[TW] > df[AMBIENT]).mean(),

    "Tt > ambient":
        (df[TT] > df[AMBIENT]).mean(),

    "Ty > ambient":
        (df[TY] > df[AMBIENT]).mean()
}

for key, value in thermal_ordering.items():

    print(
        f"{key:20s}: "
        f"{value:.4f}"
    )

pd.DataFrame(
    {
        "condition": thermal_ordering.keys(),
        "fraction": thermal_ordering.values()
    }
).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_ordering.csv"
    ),
    index=False
)

# ============================================================
# 15. ELECTRICAL DYNAMICS
# ============================================================

print("\n" + "=" * 80)
print("ELECTRICAL DYNAMICS")
print("=" * 80)

df["did_dt"] = (
    df.groupby(PROFILE_COL)[ID]
      .diff()
)

df["diq_dt"] = (
    df.groupby(PROFILE_COL)[IQ]
      .diff()
)

if TIME_COL is not None:

    valid_dt = (
        df["_dt"].notna()
        &
        (df["_dt"] > 0)
    )

    df.loc[valid_dt, "did_dt"] /= (
        df.loc[valid_dt, "_dt"]
    )

    df.loc[valid_dt, "diq_dt"] /= (
        df.loc[valid_dt, "_dt"]
    )

    df.loc[
        ~valid_dt,
        ["did_dt", "diq_dt"]
    ] = np.nan

electrical_dynamic_columns = [
    ID,
    IQ,
    UD,
    UQ,
    SPEED,
    "did_dt",
    "diq_dt",
    TORQUE
]

electrical_dynamic_stats = (
    df[electrical_dynamic_columns]
    .describe()
    .T[
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    ]
)

print(
    electrical_dynamic_stats
)

electrical_dynamic_stats.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "electrical_dynamics.csv"
    )
)

# ============================================================
# 16. ELECTRICAL CORRELATION
# ============================================================

print("\n" + "=" * 80)
print("ELECTRICAL CORRELATION MATRIX")
print("=" * 80)

electrical_columns = [
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    "did_dt",
    "diq_dt",
    TORQUE
]

electrical_corr = (
    df[electrical_columns]
    .corr()
)

print(
    electrical_corr.round(4)
)

electrical_corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "electrical_correlation_matrix.csv"
    )
)

# ============================================================
# 17. PROFILE-LEVEL THERMAL / COPPER LOSS RANGES
# ============================================================

print("\n" + "=" * 80)
print("PROFILE-LEVEL THERMAL / COPPER-LOSS RANGES")
print("=" * 80)

profile_ranges = (
    df.groupby(PROFILE_COL)
      .agg(
          Tw_range=(
              TW,
              lambda x: x.max() - x.min()
          ),

          Tt_range=(
              TT,
              lambda x: x.max() - x.min()
          ),

          Ty_range=(
              TY,
              lambda x: x.max() - x.min()
          ),

          Tpm_range=(
              TPM,
              lambda x: x.max() - x.min()
          ),

          Pcu_range=(
              "P_copper",
              lambda x: x.max() - x.min()
          )
      )
      .reset_index()
)

print(
    profile_ranges.to_string(
        index=False
    )
)

profile_ranges.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "profile_thermal_copper_ranges.csv"
    ),
    index=False
)

# ============================================================
# 18. PROFILE-LEVEL CORRELATIONS
# ============================================================

print("\n" + "=" * 80)
print("PROFILE-LEVEL PHYSICS CORRELATIONS")
print("=" * 80)

profile_corr = (
    profile_ranges.drop(
        columns=[PROFILE_COL]
    )
    .corr()
)

print(
    profile_corr.round(4)
)

profile_corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "profile_level_correlations.csv"
    )
)

# ============================================================
# 19. POWER BALANCE DIAGNOSTIC
# ============================================================
#
# Electrical power:
#       P_e = 1.5*(ud*id + uq*iq)
#
# Mechanical:
#       P_m = torque * omega
#
# Copper:
#       P_cu = Rs*(id^2 + iq^2)
#       OR current-squared proxy if Rs unavailable.
#
# Residual:
#
#       P_res = P_e - P_m - P_cu
#
# HOWEVER:
# This is only a diagnostic because:
#
# - iron losses
# - switching losses
# - mechanical losses
# - inverter losses
# - measurement noise
# - dq scaling convention
# - sign conventions
#
# are not explicitly available.
# ============================================================

print("\n" + "=" * 80)
print("ELECTRICAL / MECHANICAL POWER DIAGNOSTIC")
print("=" * 80)

power_columns = [
    "P_electrical",
    "P_mechanical",
    "P_copper"
]

power_stats = (
    df[power_columns]
    .describe()
    .T[
        [
            "mean",
            "std",
            "min",
            "max"
        ]
    ]
)

print(
    power_stats
)

power_stats.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "power_statistics.csv"
    )
)

# ============================================================
# 20. POWER BALANCE RESIDUAL
# ============================================================

df["power_balance_residual"] = (
    df["P_electrical"]
    -
    df["P_mechanical"]
    -
    df["P_copper"]
)

print("\n" + "=" * 80)
print("SIMPLE POWER BALANCE RESIDUAL")
print("=" * 80)

residual_stats = (
    df["power_balance_residual"]
    .describe()
)

print(
    residual_stats
)

# ============================================================
# 21. ROBUST RELATIVE POWER ERROR
# ============================================================
#
# DO NOT calculate:
#
# residual / P_electrical
#
# directly because P_electrical can approach zero.
#
# Instead use:
#
# |residual| /
# max(|P_electrical|, |P_mechanical|, epsilon)
#
# This prevents huge artificial percentages.
# ============================================================

epsilon = 1.0

denominator = np.maximum(
    np.maximum(
        np.abs(df["P_electrical"]),
        np.abs(df["P_mechanical"])
    ),
    epsilon
)

df["relative_power_error"] = (
    np.abs(
        df["power_balance_residual"]
    )
    /
    denominator
)

mean_relative_power_error = (
    df["relative_power_error"]
    .mean()
)

median_relative_power_error = (
    df["relative_power_error"]
    .median()
)

p90_relative_power_error = (
    df["relative_power_error"]
    .quantile(0.90)
)

p95_relative_power_error = (
    df["relative_power_error"]
    .quantile(0.95)
)

print(
    f"\nMean robust relative power error: "
    f"{mean_relative_power_error:.6f}"
)

print(
    f"Median robust relative power error: "
    f"{median_relative_power_error:.6f}"
)

print(
    f"90th percentile relative error: "
    f"{p90_relative_power_error:.6f}"
)

print(
    f"95th percentile relative error: "
    f"{p95_relative_power_error:.6f}"
)

# ============================================================
# 22. PHYSICS FEATURE CORRELATION
# ============================================================

print("\n" + "=" * 80)
print("PHYSICS FEATURE CORRELATION WITH THERMAL TARGETS")
print("=" * 80)

physics_features = [
    UD,
    UQ,
    ID,
    IQ,
    SPEED,
    TORQUE,
    "P_copper",
    "P_electrical",
    "P_mechanical"
]

thermal_targets = [
    TW,
    TT,
    TY,
    TPM
]

physics_target_corr = pd.DataFrame(
    index=physics_features,
    columns=thermal_targets,
    dtype=float
)

for feature in physics_features:

    for target in thermal_targets:

        physics_target_corr.loc[
            feature,
            target
        ] = df[feature].corr(
            df[target]
        )

print(
    physics_target_corr.round(4)
)

physics_target_corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "physics_feature_thermal_target_correlations.csv"
    )
)

# ============================================================
# 23. PROFILE VARIABILITY
# ============================================================

print("\n" + "=" * 80)
print("PROFILE VARIABILITY")
print("=" * 80)

profile_summary = (
    df.groupby(PROFILE_COL)
      .agg(
          Tw_mean=(TW, "mean"),
          Tw_std=(TW, "std"),

          Tt_mean=(TT, "mean"),
          Tt_std=(TT, "std"),

          Ty_mean=(TY, "mean"),
          Ty_std=(TY, "std"),

          Tpm_mean=(TPM, "mean"),
          Tpm_std=(TPM, "std"),

          Pcu_mean=("P_copper", "mean"),
          Pcu_std=("P_copper", "std"),

          speed_mean=(SPEED, "mean"),
          torque_mean=(TORQUE, "mean")
      )
      .reset_index()
)

print(
    profile_summary.head(20)
)

profile_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "profile_summary.csv"
    ),
    index=False
)

# ============================================================
# 24. CHECK PHYSICAL TEMPERATURE RELATIONSHIPS
# ============================================================

print("\n" + "=" * 80)
print("TEMPERATURE RELATIONSHIP CHECK")
print("=" * 80)

temperature_relationships = {

    "Tw > Tt":
        (df[TW] > df[TT]).mean(),

    "Tt > Ty":
        (df[TT] > df[TY]).mean(),

    "Tw > Ty":
        (df[TW] > df[TY]).mean(),

    "Tw > coolant":
        (df[TW] > df[COOLANT]).mean(),

    "Tt > coolant":
        (df[TT] > df[COOLANT]).mean(),

    "Ty > coolant":
        (df[TY] > df[COOLANT]).mean(),

    "Tw > ambient":
        (df[TW] > df[AMBIENT]).mean(),

    "Tt > ambient":
        (df[TT] > df[AMBIENT]).mean(),

    "Ty > ambient":
        (df[TY] > df[AMBIENT]).mean()
}

for key, value in temperature_relationships.items():

    print(
        f"{key:20s}: {value:.4f}"
    )

# ============================================================
# 25. PROFILE RANGE SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GLOBAL THERMAL / COPPER LOSS RANGES")
print("=" * 80)

global_ranges = {

    "stator_winding_range":
        df[TW].max() - df[TW].min(),

    "stator_tooth_range":
        df[TT].max() - df[TT].min(),

    "stator_yoke_range":
        df[TY].max() - df[TY].min(),

    "pm_range":
        df[TPM].max() - df[TPM].min(),

    "coolant_range":
        df[COOLANT].max() - df[COOLANT].min(),

    "ambient_range":
        df[AMBIENT].max() - df[AMBIENT].min(),

    "P_copper_range":
        df["P_copper"].max()
        -
        df["P_copper"].min()
}

for key, value in global_ranges.items():

    print(
        f"{key:30s}: {value:.6f}"
    )

# ============================================================
# 26. PHYSICS DEFENSIBILITY SCORECARD
# ============================================================

print("\n" + "=" * 80)
print("PHYSICS DEFENSIBILITY SCORECARD")
print("=" * 80)

scorecard = []

# ------------------------------------------------------------
# A. Thermal coupling
# ------------------------------------------------------------

thermal_corr_pairs = [
    (TW, TT),
    (TT, TY),
    (TY, TPM),
    (TW, TPM)
]

thermal_pair_values = []

for a, b in thermal_corr_pairs:

    thermal_pair_values.append(
        abs(
            df[a].corr(df[b])
        )
    )

thermal_coupling_score = np.mean(
    thermal_pair_values
)

scorecard.append(
    {
        "criterion":
            "Thermal subsystem coupling",

        "value":
            thermal_coupling_score,

        "interpretation":
            (
                "Strong"
                if thermal_coupling_score >= 0.7
                else
                "Moderate"
                if thermal_coupling_score >= 0.4
                else
                "Weak"
            )
    }
)

# ------------------------------------------------------------
# B. Electrical-mechanical coupling
# ------------------------------------------------------------

iq_torque_corr = abs(
    df[IQ].corr(
        df[TORQUE]
    )
)

scorecard.append(
    {
        "criterion":
            "Iq-torque coupling",

        "value":
            iq_torque_corr,

        "interpretation":
            (
                "Strong"
                if iq_torque_corr >= 0.7
                else
                "Moderate"
                if iq_torque_corr >= 0.4
                else
                "Weak"
            )
    }
)

# ------------------------------------------------------------
# C. Copper-loss / winding temperature coupling
# ------------------------------------------------------------

pcu_tw_corr = abs(
    df["P_copper"].corr(
        df[TW]
    )
)

scorecard.append(
    {
        "criterion":
            "Copper-loss / winding-temperature coupling",

        "value":
            pcu_tw_corr,

        "interpretation":
            (
                "Strong"
                if pcu_tw_corr >= 0.7
                else
                "Moderate"
                if pcu_tw_corr >= 0.4
                else
                "Weak"
            )
    }
)

# ------------------------------------------------------------
# D. Thermal dynamics
# ------------------------------------------------------------

dynamic_values = [
    abs(v)
    for v in copper_dynamic_results.values()
    if pd.notna(v)
]

dynamic_score = (
    np.mean(dynamic_values)
    if dynamic_values
    else np.nan
)

scorecard.append(
    {
        "criterion":
            "Copper-loss / thermal-dynamics correlation",

        "value":
            dynamic_score,

        "interpretation":
            (
                "Strong"
                if pd.notna(dynamic_score)
                and dynamic_score >= 0.5
                else
                "Moderate"
                if pd.notna(dynamic_score)
                and dynamic_score >= 0.2
                else
                "Weak / not established"
            )
    }
)

# ------------------------------------------------------------
# E. Thermal ordering
# ------------------------------------------------------------

ordering_score = np.mean(
    [
        thermal_ordering["Tw > coolant"],
        thermal_ordering["Tt > coolant"],
        thermal_ordering["Ty > coolant"]
    ]
)

scorecard.append(
    {
        "criterion":
            "Thermal ordering consistency",

        "value":
            ordering_score,

        "interpretation":
            (
                "Strong"
                if ordering_score >= 0.9
                else
                "Moderate"
                if ordering_score >= 0.75
                else
                "Weak"
            )
    }
)

scorecard_df = pd.DataFrame(
    scorecard
)

print(
    scorecard_df.to_string(
        index=False
    )
)

scorecard_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "physics_defensibility_scorecard.csv"
    ),
    index=False
)

# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("FINAL PHYSICS DATASET SUMMARY")
print("=" * 80)

final_summary = {

    "n_records":
        len(df),

    "n_profiles":
        df[PROFILE_COL].nunique(),

    "temperature_range_winding":
        df[TW].max() - df[TW].min(),

    "temperature_range_tooth":
        df[TT].max() - df[TT].min(),

    "temperature_range_yoke":
        df[TY].max() - df[TY].min(),

    "temperature_range_pm":
        df[TPM].max() - df[TPM].min(),

    "copper_loss_range":
        df["P_copper"].max()
        -
        df["P_copper"].min(),

    "corr_Pcu_Tw":
        df["P_copper"].corr(df[TW]),

    "corr_Pcu_dTw":
        copper_dynamic_results["dTw_dt"],

    "corr_Pcu_dTT":
        copper_dynamic_results["dTT_dt"],

    "corr_Pcu_dTY":
        copper_dynamic_results["dTY_dt"],

    "corr_Pcu_dTPM":
        copper_dynamic_results["dTPM_dt"],

    "median_relative_power_error":
        median_relative_power_error,

    "p90_relative_power_error":
        p90_relative_power_error,

    "p95_relative_power_error":
        p95_relative_power_error,

    "thermal_coupling_score":
        thermal_coupling_score,

    "iq_torque_correlation":
        iq_torque_corr,

    "derivative_basis":
        derivative_basis,

    "copper_loss_definition":
        copper_definition
}

for key, value in final_summary.items():

    print(
        f"{key:40s}: {value}"
    )

final_summary_df = pd.DataFrame(
    {
        "metric": final_summary.keys(),
        "value": final_summary.values()
    }
)

final_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "final_physics_summary.csv"
    ),
    index=False
)

# ============================================================
# 28. SAVE ANALYSIS DATASET
# ============================================================

analysis_columns = [
    PROFILE_COL,

    UD,
    UQ,
    ID,
    IQ,

    SPEED,
    TORQUE,

    TW,
    TT,
    TY,
    TPM,

    COOLANT,
    AMBIENT,

    "P_dq_raw",
    "P_electrical",
    "P_mechanical",
    "P_copper",

    "dTw",
    "dTt_y",
    "dTt_pm",
    "dTy_c",
    "dTy_a",

    "dTw_dt",
    "dTT_dt",
    "dTY_dt",
    "dTPM_dt",

    "did_dt",
    "diq_dt",

    "power_balance_residual",
    "relative_power_error"
]

analysis_columns = [
    c for c in analysis_columns
    if c in df.columns
]

analysis_df = df[analysis_columns].copy()

analysis_path = os.path.join(
    OUTPUT_DIR,
    "pmsm_physics_analysis_dataset.csv"
)

analysis_df.to_csv(
    analysis_path,
    index=False
)

# ============================================================
# 29. COMPLETION
# ============================================================

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print(
    f"\nOutput directory:\n{OUTPUT_DIR}"
)

print("\nGenerated files:")

for filename in sorted(
    os.listdir(OUTPUT_DIR)
):

    print(
        "  -",
        filename
    )

print("\n" + "=" * 80)

PMSM ELECTRO-THERMAL PHYSICS DEFENSIBILITY ANALYSIS

Loading dataset...

Dataset shape:
(1330816, 13)

Columns:
['u_q', 'coolant', 'stator_winding', 'u_d', 'stator_tooth', 'motor_speed', 'i_d', 'i_q', 'pm', 'stator_yoke', 'ambient', 'torque', 'profile_id']

All required columns are present.

Rows removed because of NaN/non-numeric values: 0
Remaining rows: 1,330,816

Number of profiles: 69

Profile size statistics:
count       69.000000
mean     19287.188406
std       9606.729772
min       2176.000000
25%      14403.000000
50%      17142.000000
75%      23761.000000
max      43971.000000
dtype: float64

TEMPORAL ORDERING CHECK

No timestamp/time column exists in the supplied dataset.
Derivatives will use CSV row order within each profile.
Therefore dT/dt should be interpreted as 'temperature change per sample', NOT physical dT/dt.

ELECTRICAL / MECHANICAL POWER FEATURES

Power features created:
P_dq_raw
P_electrical
P_copper
omega_rad_s
P_mechanical

THERMAL TEMPERATURE DIFFERENCES
   